# Лабораторная работа 5: Cross-Validation и ROC в Python — исправленная версия

Исправление касается ROC-части: Python теперь берёт только исходную таблицу `TP/FN/FP/TN/cutoff`, а не нижние вспомогательные таблицы Excel. Именно из-за этого прежний график отличался от Excel.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize

cross_validation_file = "Chapter5-HW-CrossValidation.xlsx"
roc_file = "Chapter5-roc-HW.xlsx"


## 1. Cross-validation для LDA

In [ ]:
training_sheet = "LDA-Training-Dataset"
testing_sheet = "Testing-Dataset"

train_raw = pd.read_excel(cross_validation_file, sheet_name=training_sheet)
test_raw = pd.read_excel(cross_validation_file, sheet_name=testing_sheet)

features = [f"C{i}" for i in range(1, 14)]

train = train_raw[["Vendor"] + features].dropna().copy()
test = test_raw[["Vendor"] + features].dropna().copy()

class_to_num = {"V1": 0, "V2": 1, "V3": 2}

train["Y"] = train["Vendor"].map(class_to_num)
test["Y"] = test["Vendor"].map(class_to_num)

X_train = train[features].astype(float).values
y_train = train["Y"].astype(float).values
X_test = test[features].astype(float).values

X_train_design = np.column_stack([np.ones(len(X_train)), X_train])
X_test_design = np.column_stack([np.ones(len(X_test)), X_test])

print("Training objects:", len(train))
print("Testing objects:", len(test))


In [ ]:
def calculate_predictions(X_design, coefficients):
    return X_design @ coefficients

def calculate_means_counts_cutoffs(predictions, y):
    temp = pd.DataFrame({"Y": y, "Prediction": predictions})
    means = temp.groupby("Y")["Prediction"].mean()
    counts = temp.groupby("Y")["Prediction"].count()
    cutoff_01 = (means.loc[0] * counts.loc[0] + means.loc[1] * counts.loc[1]) / (counts.loc[0] + counts.loc[1])
    cutoff_12 = (means.loc[1] * counts.loc[1] + means.loc[2] * counts.loc[2]) / (counts.loc[1] + counts.loc[2])
    return means, counts, [cutoff_01, cutoff_12]

def classify_by_cutoffs(predictions, cutoffs):
    out = []
    for value in predictions:
        if value < cutoffs[0]:
            out.append("V1")
        elif value < cutoffs[1]:
            out.append("V2")
        else:
            out.append("V3")
    return np.array(out)

def count_difference(real_classes, predicted_classes):
    return int((real_classes.values != predicted_classes).sum())

def calculate_lda_metrics(coefficients, X_design, y):
    predictions = calculate_predictions(X_design, coefficients)
    means, counts, cutoffs = calculate_means_counts_cutoffs(predictions, y)
    inter = ((means - means.mean()) ** 2).sum()
    within = 0
    for c in [0, 1, 2]:
        vals = predictions[y == c]
        within += ((vals - means.loc[c]) ** 2).sum()
    return {
        "predictions": predictions,
        "means": means,
        "counts": counts,
        "cutoffs": cutoffs,
        "inter_group_variance": inter,
        "within_group_variance": within,
        "ratio": inter / within
    }


In [ ]:
linest_coefficients = np.linalg.lstsq(X_train_design, y_train, rcond=None)[0]
linest_metrics = calculate_lda_metrics(linest_coefficients, X_train_design, y_train)
linest_cutoffs = linest_metrics["cutoffs"]

linest_train_classes = classify_by_cutoffs(linest_metrics["predictions"], linest_cutoffs)
linest_train_missed = count_difference(train["Vendor"], linest_train_classes)
linest_train_accuracy = 1 - linest_train_missed / len(train)

linest_test_predictions = calculate_predictions(X_test_design, linest_coefficients)
linest_test_classes = classify_by_cutoffs(linest_test_predictions, linest_cutoffs)
linest_test_missed = count_difference(test["Vendor"], linest_test_classes)
linest_test_accuracy = 1 - linest_test_missed / len(test)

print("LINEST training missed:", linest_train_missed)
print("LINEST training accuracy:", f"{linest_train_accuracy:.2%}")
print("LINEST cross-validation missed:", linest_test_missed)
print("LINEST cross-validation accuracy:", f"{linest_test_accuracy:.2%}")
print("LINEST ratio:", linest_metrics["ratio"])


In [ ]:
def objective(coefficients):
    return -calculate_lda_metrics(coefficients, X_train_design, y_train)["ratio"]

bounds = [(-10, 10)] * len(linest_coefficients)

result = minimize(
    objective,
    linest_coefficients,
    method="L-BFGS-B",
    bounds=bounds,
    options={"maxiter": 10000}
)

solver_coefficients = result.x
solver_metrics = calculate_lda_metrics(solver_coefficients, X_train_design, y_train)
solver_cutoffs = solver_metrics["cutoffs"]

solver_train_classes = classify_by_cutoffs(solver_metrics["predictions"], solver_cutoffs)
solver_train_missed = count_difference(train["Vendor"], solver_train_classes)
solver_train_accuracy = 1 - solver_train_missed / len(train)

solver_test_predictions = calculate_predictions(X_test_design, solver_coefficients)
solver_test_classes = classify_by_cutoffs(solver_test_predictions, solver_cutoffs)
solver_test_missed = count_difference(test["Vendor"], solver_test_classes)
solver_test_accuracy = 1 - solver_test_missed / len(test)

coef_table = pd.DataFrame({
    "Coefficient": ["b"] + [f"w{i}" for i in range(1, 14)],
    "LINEST": linest_coefficients,
    "Solver-like": solver_coefficients
})

summary = pd.DataFrame({
    "Model": ["LINEST", "Solver-like"],
    "Training missed": [linest_train_missed, solver_train_missed],
    "Training accuracy": [linest_train_accuracy, solver_train_accuracy],
    "Testing missed": [linest_test_missed, solver_test_missed],
    "Cross-validation accuracy": [linest_test_accuracy, solver_test_accuracy],
    "Inter/within ratio": [linest_metrics["ratio"], solver_metrics["ratio"]]
})

display(coef_table)
display(summary)


## 2. ROC-анализ

Главное исправление: берём только строки исходной таблицы, где `TP`, `FN`, `FP`, `TN` — настоящие большие значения. Нижние вспомогательные таблицы Excel отбрасываются.


In [ ]:
roc_raw = pd.read_excel(roc_file)

roc = roc_raw.iloc[:, :5].copy()
roc.columns = ["TP", "FN", "FP", "TN", "cutoff"]

for col in ["TP", "FN", "FP", "TN", "cutoff"]:
    roc[col] = pd.to_numeric(roc[col], errors="coerce")

roc = roc.dropna(subset=["TP", "FN", "FP", "TN", "cutoff"]).copy()

# Фильтр защищает от попадания нижней ROC-таблицы.
roc = roc[
    (roc["TP"] > 100) &
    (roc["FN"] > 0) &
    (roc["FP"] > 0) &
    (roc["TN"] > 100) &
    (roc["cutoff"] >= 0)
].copy()

roc["sensitivity"] = roc["TP"] / (roc["TP"] + roc["FN"])
roc["specificity"] = roc["TN"] / (roc["TN"] + roc["FP"])

roc


In [ ]:
roc_for_plot = roc.sort_values("cutoff")

plt.figure()
plt.plot(roc_for_plot["cutoff"], roc_for_plot["sensitivity"], marker="o", label="sensitivity")
plt.plot(roc_for_plot["cutoff"], roc_for_plot["specificity"], marker="o", label="specificity")
plt.xlabel("cutoff")
plt.ylabel("value")
plt.title("Sensitivity and specificity vs cutoff")
plt.ylim(0, 1.05)
plt.grid(True)
plt.legend()
plt.show()


In [ ]:
roc_curve_data = roc[["cutoff", "sensitivity", "specificity"]].copy()
roc_curve_data["100-specificity"] = 100 - np.round(roc_curve_data["specificity"] * 100, 0)
roc_curve_data["sensitivity_percent"] = np.round(roc_curve_data["sensitivity"] * 100, 0)

roc_curve_data = roc_curve_data.sort_values("100-specificity")

zero_point = pd.DataFrame({
    "cutoff": [np.nan],
    "sensitivity": [0],
    "specificity": [1],
    "100-specificity": [0],
    "sensitivity_percent": [0]
})

roc_curve_data = pd.concat([zero_point, roc_curve_data], ignore_index=True)

roc_curve_data


In [ ]:
plt.figure()
plt.plot(
    roc_curve_data["100-specificity"],
    roc_curve_data["sensitivity_percent"],
    marker="o"
)
plt.xlabel("100-specificity")
plt.ylabel("Sensitivity")
plt.title("ROC Curve")
plt.xlim(0, 100)
plt.ylim(0, 100)
plt.grid(True)
plt.show()


In [ ]:
auc = np.trapz(
    roc_curve_data["sensitivity_percent"],
    roc_curve_data["100-specificity"]
) / 10000

print("Approximate AUC:", auc)


In [ ]:
output_file = "Chapter5_Python_Results_FIXED.xlsx"

with pd.ExcelWriter(output_file) as writer:
    summary.to_excel(writer, sheet_name="CV summary", index=False)
    coef_table.to_excel(writer, sheet_name="Coefficients", index=False)
    roc.to_excel(writer, sheet_name="ROC source cleaned", index=False)
    roc_curve_data.to_excel(writer, sheet_name="ROC curve data", index=False)

print(f"Results saved to: {output_file}")
